# Aritmética modular y primos

Trabajar módulo $m$ mantiene los números pequeños: $(a + b) \bmod m$ y $(a \cdot b) \bmod m$ se pueden calcular reduciendo antes cada operando. La criba de Eratóstenes obtiene todos los primos hasta $n$ en $O(n \log \log n)$; la exponenciación rápida calcula $a^e \bmod m$ en $O(\log e)$ multiplicaciones; y, si $p$ es primo, el pequeño teorema de Fermat ($a^{p-1} \equiv 1 \pmod p$) da el inverso modular $a^{-1} \equiv a^{p-2}$.

## 1. Criba de Eratóstenes

Implementar la criba hasta $n$, comprobarla contra la división por tentativa y contar los primos menores que $10^6$.

In [1]:
import math
import random
import time

def criba(n):
    es_primo = bytearray([1]) * (n + 1)
    es_primo[:2] = b"\x00\x00"
    for i in range(2, math.isqrt(n) + 1):
        if es_primo[i]:
            es_primo[i * i::i] = bytes(len(range(i * i, n + 1, i)))
    return [i for i in range(n + 1) if es_primo[i]]

def es_primo_tentativa(n):
    return n >= 2 and all(n % d for d in range(2, math.isqrt(n) + 1))

assert criba(1000) == [i for i in range(1001) if es_primo_tentativa(i)]
inicio = time.perf_counter()
primos = criba(10**6)
print(len(primos), "primos menores que un millón en", round(time.perf_counter() - inicio, 3), "s")
print(primos[:15], "...", primos[-3:])

78498 primos menores que un millón en 0.022 s
[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47] ... [999961, 999979, 999983]


## 2. Criba segmentada

Obtener los primos de un intervalo alto $[L, R]$ (por ejemplo, a partir de $10^{12}$) sin cribar desde 0: basta tachar en el intervalo los múltiplos de los primos hasta $\sqrt{R}$.

In [2]:
def criba_segmentada(L, R):
    marca = bytearray([1]) * (R - L + 1)
    for p in criba(math.isqrt(R)):
        inicio = max(p * p, (L + p - 1) // p * p)
        marca[inicio - L::p] = bytes(len(range(inicio, R + 1, p)))
    return [L + i for i, m in enumerate(marca) if m and L + i >= 2]

L = 10**12
encontrados = criba_segmentada(L, L + 1000)
print(encontrados)
assert all(es_primo_tentativa(p) for p in encontrados[:5])
assert criba_segmentada(2, 5000) == criba(5000)

[1000000000039, 1000000000061, 1000000000063, 1000000000091, 1000000000121, 1000000000163, 1000000000169, 1000000000177, 1000000000189, 1000000000193, 1000000000211, 1000000000271, 1000000000303, 1000000000331, 1000000000333, 1000000000339, 1000000000459, 1000000000471, 1000000000537, 1000000000543, 1000000000547, 1000000000561, 1000000000609, 1000000000661, 1000000000669, 1000000000721, 1000000000751, 1000000000787, 1000000000789, 1000000000799, 1000000000841, 1000000000903, 1000000000921, 1000000000931, 1000000000933, 1000000000949, 1000000000997]


## 3. Factorización

Factorizar con la criba del menor factor primo (para muchos números pequeños) y con división por tentativa (para uno grande), y comprobar que el producto de los factores devuelve el número.

In [3]:
def menor_factor(n):
    mf = list(range(n + 1))
    for i in range(2, math.isqrt(n) + 1):
        if mf[i] == i:
            for j in range(i * i, n + 1, i):
                if mf[j] == j:
                    mf[j] = i
    return mf

MF = menor_factor(10**5)

def factoriza_rapido(n):
    factores = []
    while n > 1:
        factores.append(MF[n])
        n //= MF[n]
    return factores

def factoriza(n):
    factores, d = [], 2
    while d * d <= n:
        while n % d == 0:
            factores.append(d)
            n //= d
        d += 1
    return factores + ([n] if n > 1 else [])

print(factoriza_rapido(84_000), factoriza(600_851_475_143))
random.seed(1)
for _ in range(1000):
    n = random.randint(2, 10**5)
    assert factoriza_rapido(n) == factoriza(n) and math.prod(factoriza(n)) == n

[2, 2, 2, 2, 2, 3, 5, 5, 5, 7] [71, 839, 1471, 6857]


## 4. Exponenciación modular e inverso

Implementar la exponenciación rápida $a^e \bmod m$ por cuadrados sucesivos, compararla con `pow(a, e, m)` y calcular inversos modulares con Fermat y con `pow(a, -1, m)`.

In [4]:
def potencia_mod(a, e, m):
    resultado, a = 1, a % m
    while e:
        if e & 1:
            resultado = resultado * a % m
        a = a * a % m
        e >>= 1
    return resultado

MOD = 1_000_000_007
for _ in range(1000):
    a, e, m = random.randint(0, 10**12), random.randint(0, 10**18), random.randint(1, 10**9)
    assert potencia_mod(a, e, m) == pow(a, e, m)

a = 123_456_789
inverso = potencia_mod(a, MOD - 2, MOD)
print(inverso, inverso == pow(a, -1, MOD), a * inverso % MOD)

18633540 True 1


## 5. Combinatoria módulo p

Calcular $\binom{n}{k} \bmod p$ para $n$ hasta $10^6$ precalculando factoriales e inversos de factoriales, y comprobarlo contra `math.comb`.

In [5]:
N = 10**6
fact = [1] * (N + 1)
for i in range(1, N + 1):
    fact[i] = fact[i - 1] * i % MOD
inv_fact = [1] * (N + 1)
inv_fact[N] = pow(fact[N], MOD - 2, MOD)
for i in range(N, 0, -1):
    inv_fact[i - 1] = inv_fact[i] * i % MOD

def comb_mod(n, k):
    return 0 if k < 0 or k > n else fact[n] * inv_fact[k] % MOD * inv_fact[n - k] % MOD

for _ in range(200):
    n = random.randint(0, 2000)
    k = random.randint(0, n)
    assert comb_mod(n, k) == math.comb(n, k) % MOD
print(comb_mod(10**6, 500_000))

996692777


## 6. Miller-Rabin

Implementar el test de primalidad de Miller-Rabin. Con las bases $2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37$ es determinista para todo $n < 3.3 \cdot 10^{24}$, que incluye los enteros de 64 bits. Comprobarlo contra la criba y con números de Carmichael, que engañan al test de Fermat.

In [6]:
BASES = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)

def miller_rabin(n):
    if n < 2:
        return False
    for p in BASES:
        if n % p == 0:
            return n == p
    d, s = n - 1, 0
    while d % 2 == 0:
        d //= 2
        s += 1
    for a in BASES:
        x = pow(a, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = x * x % n
            if x == n - 1:
                break
        else:
            return False
    return True

primos_set = set(criba(10**5))
assert all(miller_rabin(n) == (n in primos_set) for n in range(10**5))

carmichael = [561, 1105, 1729, 2465, 2821, 6601]
print("Fermat con base 2 dice primo:", [n for n in carmichael if pow(2, n - 1, n) == 1])
print("Miller-Rabin dice primo:", [n for n in carmichael if miller_rabin(n)])
print(miller_rabin(2**61 - 1), miller_rabin(2**64 - 59), miller_rabin(2**64 - 1))

Fermat con base 2 dice primo: [561, 1105, 1729, 2465, 2821, 6601]
Miller-Rabin dice primo: []
True True False
